# Bypass vs. baseline

Does routing small flows on shorter paths cut their tail latency, what does it cost other traffic, and at what load does the cost outweigh the gain?

Reads the baseline and bypass runs of the sweep set by `SWEEP` in `common.py`. It only compares a load and seed once both the baseline and a bypass mode have finished it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import common

common.setup_plots()
pd.options.display.float_format = "{:,.2f}".format

baseline = common.load_runs(common.BASELINE_EXPERIMENT)
bypass = common.load_runs(common.BYPASS_EXPERIMENT)

# Keep only loads and seeds where the baseline and a bypass run both finished.
pairs = {(r.load, r.seed) for r in baseline} & {(r.load, r.seed) for r in bypass}
runs = [r for r in baseline + bypass if (r.load, r.seed) in pairs]
ready = bool(pairs)
if ready:
    flows = common.load_flows(runs)
    links = common.load_links(runs)
    print(f"Comparing {len(pairs)} load and seed pairs: {sorted(pairs)}")
    print(f"Modes: {sorted({r.bypass for r in runs})}")
else:
    print(f"Nothing to compare yet: {len(baseline)} baseline runs and {len(bypass)} bypass runs share no load and seed.")
    print(f"Run experiments/{common.BASELINE_EXPERIMENT}.sh and experiments/{common.BYPASS_EXPERIMENT}.sh.")

## Are the runs healthy and comparable?

Every mode must run the same flows, or the comparison means nothing.

In [2]:
if ready:
    display(common.runs_table(runs))
    identity = ["flow", "source_router", "destination_router", "bytes", "start_ns"]
    mismatches = 0
    for key, group in flows.groupby(["load", "seed"]):
        reference = group[group["bypass"] == "none"][identity].reset_index(drop=True)
        for mode, other in group.groupby("bypass"):
            if not reference.equals(other[identity].reset_index(drop=True)):
                print(f"Different flows at load and seed {key}, mode {mode}")
                mismatches += 1
    print("All modes run the same flows" if mismatches == 0 else f"{mismatches} mismatches")

## H1: Does the bypass cut small-flow latency?

Expect the biggest gain for small flows between adjacent routers, since they cross one queue instead of two to five.

In [3]:
if ready:
    display(common.fct_table(flows, ["class", "load", "bypass"]))

In [4]:
if ready:
    common.plot_metric_by_mode(flows, "p50", "Median FCT")
    common.plot_metric_by_mode(flows, "p99", "p99 FCT")

### Flow by flow

Since every mode runs the same flows, each flow can be compared with itself. A ratio below 1 means the bypass made that flow faster.

In [5]:
if ready:
    keys = ["load", "seed", "flow"]
    reference = flows[flows["bypass"] == "none"][keys + ["fct_us"]].rename(columns={"fct_us": "baseline_us"})
    paired = flows[flows["bypass"] != "none"].merge(reference, on=keys)
    paired["ratio"] = paired["fct_us"] / paired["baseline_us"]
    display(
        paired.groupby(["class", "load", "bypass"]).agg(
            median_ratio=("ratio", "median"),
            faster=("ratio", lambda r: (r < 1).mean()),
            slower=("ratio", lambda r: (r > 1).mean()),
        )
    )

## H2: What does it cost, and where is the crossover?

Each ratio is a bypass mode's p99 over the baseline's at the same load. Above 1 means the bypass made that class's tail worse. The crossover is the load where a small-flow ratio passes 1, and a large-flow ratio above 1 shows the cost to other traffic.

In [6]:
if ready:
    p99 = common.fct_table(flows, ["class", "load", "bypass"])["p99"].unstack("bypass")
    ratios = pd.DataFrame({mode: p99[mode] / p99["none"] for mode in p99.columns if mode != "none"})
    display(ratios)

### Hotspots

The paper warns that shortest paths could congest direct links. `hotspot_ratio` is the busiest link over the mean link.

In [7]:
if ready:
    table = common.link_table(links, ["load", "bypass"])
    display(table)
    figure, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
    for axis, column, label in zip(axes, ["max_utilization", "hotspot_ratio"], ["Busiest link's utilization", "Busiest link / mean link"]):
        for mode in common.MODES:
            group = table.xs(mode, level="bypass").reset_index() if mode in table.index.get_level_values("bypass") else None
            if group is None:
                continue
            axis.plot(group["load"], group[column], color=common.MODE_COLORS[mode], marker=common.MODE_MARKERS[mode], label=common.MODE_NAMES[mode])
        axis.set(xlabel="Offered load", ylabel=label, ylim=(0, None))
    axes[0].legend()
    figure.tight_layout()

### Fan-in

Does the bypass concentrate a destination's traffic on fewer neighbors?

In [8]:
if ready:
    display(common.fan_in_table(links, ["load", "bypass"]))

## H3: How many flows does it reach?

An adjacent-only bypass can only help small flows between neighboring routers. Comparing all small flows shows what that limit costs against bypassing everything.

In [9]:
if ready:
    small = flows[flows["class"] != "large"]
    print(f"Adjacent share of small flows: {(small['class'] == 'small_adjacent').mean():.1%}")
    display(common.fct_table(small.assign(group="all small"), ["group", "load", "bypass"]))